# Kalman-Smoothed Latent SOH Walkthrough

This notebook ports the `ml_workspace/latent_soh` ideas into the `ml_workspace/soh_estimation` section so the SOH spike-smoothing workflow can be run and inspected interactively.

The core idea:

- treat raw BMS SOH as a noisy observation
- increase measurement noise when the estimator looks stressed
- smooth the event-level SOH sequence with a 1D Kalman filter plus RTS smoother
- compare raw SOH, smoothed SOH, and residual spike behavior

This is specifically for your use case: smoothing out observed SOH spikes rather than trusting the raw BMS SOH directly.


## Setup

By default the notebook can either:

- run the latent SOH pipeline for a plane and write fresh outputs, or
- load existing outputs from the notebook output directory


In [1]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "ml_workspace").exists() and (candidate / "data").exists():
            return candidate
    raise RuntimeError("Could not locate repo root")


def resolve_timeseries_path(repo_root: Path) -> Path:
    preferred = repo_root / "data" / "event_timeseries_corrected.parquet"
    fallback = repo_root / "data" / "event_timeseries.parquet"
    if preferred.exists():
        return preferred
    if fallback.exists():
        return fallback
    raise FileNotFoundError("Could not find event_timeseries parquet under data/")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from ml_workspace.latent_soh.build_latent_soh import build_latent_soh_labels

PLANE_ID = "166"
RUN_PIPELINE = False
COMPARE_BACKEND = True
RT_PROFILE = "balanced"
Q_DAY_SIGMA_PCT = 0.05
SPIKE_THRESHOLD_PCT = 2.0

TIMESERIES_PATH = resolve_timeseries_path(REPO_ROOT)
SPEC_PATH = REPO_ROOT / "ml_workspace" / "battery_specs.yaml"
OUTPUT_DIR = REPO_ROOT / "ml_workspace" / "soh_estimation" / "output" / f"latent_soh_plane_{PLANE_ID}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

print("Repo root:", REPO_ROOT)
print("Timeseries path:", TIMESERIES_PATH)
print("Output dir:", OUTPUT_DIR)


Repo root: /Users/benfogerty/Desktop/EPlaneCapstone/CapstoneEPlane
Timeseries path: /Users/benfogerty/Desktop/EPlaneCapstone/CapstoneEPlane/data/event_timeseries_corrected.parquet
Output dir: /Users/benfogerty/Desktop/EPlaneCapstone/CapstoneEPlane/ml_workspace/soh_estimation/output/latent_soh_plane_166


## Run or load the latent SOH pipeline

In [2]:
if RUN_PIPELINE:
    result = build_latent_soh_labels(
        plane_id=PLANE_ID,
        timeseries_path=TIMESERIES_PATH,
        spec_path=SPEC_PATH,
        output_dir=OUTPUT_DIR,
        q_day_sigma_pct=Q_DAY_SIGMA_PCT,
        spike_threshold_pct=SPIKE_THRESHOLD_PCT,
        compare_backend=COMPARE_BACKEND,
        rt_profile=RT_PROFILE,
    )
    print(json.dumps(result, indent=2))
else:
    print(f"Using existing latent SOH outputs in {OUTPUT_DIR}")


Using existing latent SOH outputs in /Users/benfogerty/Desktop/EPlaneCapstone/CapstoneEPlane/ml_workspace/soh_estimation/output/latent_soh_plane_166


## Load output tables

In [3]:
event_df = pd.read_csv(OUTPUT_DIR / "event_observation_table.csv", parse_dates=["event_datetime"])
latent_df = pd.read_csv(OUTPUT_DIR / "latent_soh_event_table.csv", parse_dates=["event_datetime"])
summary = json.loads((OUTPUT_DIR / "diagnostics" / "smoother_summary.json").read_text())
backend_summary = json.loads((OUTPUT_DIR / "diagnostics" / "backend_agreement_summary.json").read_text())
condition_summary = pd.read_csv(OUTPUT_DIR / "diagnostics" / "condition_score_summary.csv")
top_spikes = pd.read_csv(OUTPUT_DIR / "diagnostics" / "top_raw_spike_events.csv", parse_dates=["event_datetime"])
spike_summary = pd.read_csv(OUTPUT_DIR / "diagnostics" / "spike_feature_summary.csv")

print(f"Event rows: {len(event_df):,}")
print(f"Latent rows: {len(latent_df):,}")
display(pd.DataFrame(backend_summary))
display(condition_summary.round(3))


FileNotFoundError: [Errno 2] No such file or directory: '/Users/benfogerty/Desktop/EPlaneCapstone/CapstoneEPlane/ml_workspace/soh_estimation/output/latent_soh_plane_166/event_observation_table.csv'

## 1. Raw observed SOH

This is the unsmoothed label. The upward steps are the spikes we want to suppress.


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
for ax, (battery_id, group) in zip(axes, latent_df.groupby("battery_id", sort=True)):
    g = group.sort_values(["event_datetime", "flight_id"])
    ax.scatter(g["event_datetime"], g["observed_soh_pct"], s=12, alpha=0.75, label="Observed SOH")
    ax.set_title(f"Plane {PLANE_ID} Battery {battery_id}: raw event-level SOH")
    ax.set_ylabel("SOH (%)")
    ax.legend(loc="best")
fig.autofmt_xdate()
fig.tight_layout()


## 2. Condition-aware trust in the observation

`measurement_sigma_pct` is the effective observation standard deviation. Larger values mean we trust the raw BMS SOH less at that event.


In [ ]:
condition_cols = [
    "score_current",
    "score_didt",
    "score_dtemp",
    "score_soc_edge",
    "score_gap",
    "score_switch",
    "score_event_type",
    "score_missing",
    "condition_multiplier",
    "measurement_sigma_pct",
]
display(latent_df.groupby("battery_id")[condition_cols].median().round(3))

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
for ax, (battery_id, group) in zip(axes, latent_df.groupby("battery_id", sort=True)):
    g = group.sort_values(["event_datetime", "flight_id"])
    ax.plot(g["event_datetime"], g["measurement_sigma_pct"], color="#9467bd", linewidth=1.8)
    ax.set_title(f"Plane {PLANE_ID} Battery {battery_id}: measurement sigma over time")
    ax.set_ylabel("Sigma (%)")
fig.autofmt_xdate()
fig.tight_layout()


## 3. Kalman-smoothed latent SOH

This is the main comparison:

- gray: observed SOH
- blue: one-step filtered state
- red: RTS smoothed state
- green: optional PyKalman comparison


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
for ax, (battery_id, group) in zip(axes, latent_df.groupby("battery_id", sort=True)):
    g = group.sort_values(["event_datetime", "flight_id"])
    ax.plot(g["event_datetime"], g["observed_soh_pct"], color="0.75", linewidth=1.2, label="Observed SOH")
    ax.plot(g["event_datetime"], g["latent_soh_filter_pct"], color="#1f77b4", linewidth=1.2, label="Filter")
    ax.plot(g["event_datetime"], g["latent_soh_smooth_pct"], color="#d62728", linewidth=2.0, label="RTS smooth")
    if "latent_soh_pykalman_smooth_pct" in g.columns and g["latent_soh_pykalman_smooth_pct"].notna().any():
        ax.plot(g["event_datetime"], g["latent_soh_pykalman_smooth_pct"], color="#2ca02c", linewidth=1.4, label="PyKalman")
    ax.fill_between(
        g["event_datetime"],
        g["latent_soh_smooth_pct"] - 2.0 * g["latent_soh_smooth_std_pct"],
        g["latent_soh_smooth_pct"] + 2.0 * g["latent_soh_smooth_std_pct"],
        color="#d62728",
        alpha=0.15,
        linewidth=0,
    )
    ax.set_title(f"Plane {PLANE_ID} Battery {battery_id}: observed vs latent SOH")
    ax.set_ylabel("SOH (%)")
    ax.legend(loc="best")
fig.autofmt_xdate()
fig.tight_layout()


## 4. Monotone projection

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
for ax, (battery_id, group) in zip(axes, latent_df.groupby("battery_id", sort=True)):
    g = group.sort_values(["event_datetime", "flight_id"])
    ax.plot(g["event_datetime"], g["observed_soh_pct"], color="0.85", linewidth=1.0, label="Observed SOH")
    ax.plot(g["event_datetime"], g["latent_soh_smooth_pct"], color="#d62728", linewidth=1.8, label="RTS smooth")
    ax.plot(g["event_datetime"], g["latent_soh_monotone_pct"], color="#000000", linewidth=2.0, linestyle="--", label="Monotone projection")
    ax.set_title(f"Plane {PLANE_ID} Battery {battery_id}: monotone SOH option")
    ax.set_ylabel("SOH (%)")
    ax.legend(loc="best")
fig.autofmt_xdate()
fig.tight_layout()


## 5. Raw spike suppression check

In [ ]:
spike_rows = []
for battery_id, group in latent_df.groupby("battery_id", sort=True):
    g = group.sort_values(["event_datetime", "flight_id"]).copy()
    g["delta_observed"] = g["observed_soh_pct"].diff()
    g["delta_latent"] = g["latent_soh_smooth_pct"].diff()
    spike_rows.append(
        {
            "battery_id": battery_id,
            "raw_total_variation": g["delta_observed"].abs().sum(),
            "latent_total_variation": g["delta_latent"].abs().sum(),
            "raw_max_upward_jump_pct": g["delta_observed"].max(),
            "latent_max_upward_jump_pct": g["delta_latent"].max(),
        }
    )
spike_compare = pd.DataFrame(spike_rows)
display(spike_compare.round(3))

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex="col")
axes = np.atleast_2d(axes)
for row_idx, (battery_id, group) in enumerate(latent_df.groupby("battery_id", sort=True)):
    g = group.sort_values(["event_datetime", "flight_id"]).copy()
    g["delta_observed"] = g["observed_soh_pct"].diff()
    g["delta_latent"] = g["latent_soh_smooth_pct"].diff()
    axes[row_idx, 0].plot(g["event_datetime"], g["delta_observed"], color="#ff7f0e", linewidth=1.5)
    axes[row_idx, 0].axhline(SPIKE_THRESHOLD_PCT, color="#d62728", linestyle="--", linewidth=1.0)
    axes[row_idx, 0].set_title(f"Battery {battery_id}: raw SOH deltas")
    axes[row_idx, 0].set_ylabel("Delta SOH (%)")
    axes[row_idx, 1].plot(g["event_datetime"], g["delta_latent"], color="#1f77b4", linewidth=1.5)
    axes[row_idx, 1].axhline(SPIKE_THRESHOLD_PCT, color="#d62728", linestyle="--", linewidth=1.0)
    axes[row_idx, 1].set_title(f"Battery {battery_id}: latent SOH deltas")
fig.autofmt_xdate()
fig.tight_layout()


## 6. Residual diagnostics

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for row_idx, (battery_id, group) in enumerate(latent_df.groupby("battery_id", sort=True)):
    g = group.sort_values(["event_datetime", "flight_id"])
    axes[row_idx, 0].hist(g["residual_pct"].dropna(), bins=30, color="#ff7f0e", alpha=0.85)
    axes[row_idx, 0].set_title(f"Battery {battery_id}: residual histogram")
    axes[row_idx, 0].set_xlabel("Observed - latent SOH (%)")
    axes[row_idx, 1].scatter(g["score_observation_instability"], g["residual_pct"], s=14, alpha=0.55, color="#bcbd22")
    axes[row_idx, 1].set_title(f"Battery {battery_id}: residual vs instability")
    axes[row_idx, 1].set_xlabel("score_observation_instability")
    axes[row_idx, 1].set_ylabel("Residual (%)")
fig.tight_layout()


## 7. Top raw spike events

In [ ]:
display(top_spikes.head(20).round(3))
display(spike_summary.round(3))


## 8. Quantify how much smoothing reduced spike behavior

In [ ]:
smoother_rows = []
for battery_id, group in latent_df.groupby("battery_id", sort=True):
    g = group.sort_values(["event_datetime", "flight_id"]).copy()
    g["delta_observed"] = g["observed_soh_pct"].diff()
    g["delta_latent"] = g["latent_soh_smooth_pct"].diff()
    raw_spikes = int((g["delta_observed"] >= SPIKE_THRESHOLD_PCT).sum())
    latent_spikes = int((g["delta_latent"] >= SPIKE_THRESHOLD_PCT).sum())
    smoother_rows.append(
        {
            "battery_id": battery_id,
            "raw_spike_events": raw_spikes,
            "latent_spike_events": latent_spikes,
            "spike_reduction_pct": 100.0 * (raw_spikes - latent_spikes) / raw_spikes if raw_spikes else np.nan,
            "raw_total_variation": g["delta_observed"].abs().sum(),
            "latent_total_variation": g["delta_latent"].abs().sum(),
            "variation_reduction_pct": 100.0 * (g["delta_observed"].abs().sum() - g["delta_latent"].abs().sum()) / g["delta_observed"].abs().sum()
            if g["delta_observed"].abs().sum() > 0
            else np.nan,
        }
    )
smoothing_effect = pd.DataFrame(smoother_rows)
display(smoothing_effect.round(3))


## 9. Notebook interpretation

In [ ]:
print("Smoother summary:")
print(json.dumps(summary, indent=2))

print("\nInterpretation:")
for _, row in smoothing_effect.iterrows():
    print(
        f"- Battery {int(row['battery_id'])}: raw spike events {int(row['raw_spike_events'])}, "
        f"latent spike events {int(row['latent_spike_events'])}, "
        f"spike reduction {row['spike_reduction_pct']:.1f}%."
    )
print("- If the latent spike count and total variation both drop materially, the Kalman smoother is doing the intended job: preserving trend while downweighting implausible raw SOH jumps.")
print("- If measurement sigma rises around high-instability events, the condition-aware R_t logic is actively reducing trust in those observations.")
print("- The monotone projection is optional. It is useful if you want a strictly non-increasing SOH label for downstream training.")
